<div dir="rtl" align="right">

<h1>پاک‌سازی داده</h1>

<p>توی این نوت‌بوک، یک دیتاست خام رو تمیز و آماده‌ی تحلیل می‌کنیم.</p>

<p>کارهایی که انجام می‌دیم:</p>

<ul>
  <li>ستون‌های اضافی رو حذف می‌کنیم و ستون‌ها رو بر اساس نوعشون دسته‌بندی می‌کنیم.</li>
  <li>متن‌ها رو به عدد و boolean تبدیل می‌کنیم و تاریخ‌ها رو قابل استفاده می‌کنیم.</li>
  <li>مقادیر فارسی رو به معادل انگلیسی ترجمه می‌کنیم و مقادیر جایگزینی مثل <code>"unselect"</code> رو به مقدار گم‌شده تبدیل می‌کنیم.</li>
  <li>از توی متن‌ها، مقادیر مفیدی مثل متراژ و امکانات رو بیرون می‌کشیم.</li>
  <li>خانواده‌ی معامله رو مشخص می‌کنیم و برای هرکدوم مقدار هدف درست رو کنار می‌ذاریم.</li>
  <li>داده‌های پرت رو با روش‌های مناسب هر ستون حذف می‌کنیم.</li>
  <li>در آخر هم مختصات جغرافیایی رو فیلتر می‌کنیم و یک خلاصه‌ی عددی از داده می‌گیریم.</li>
</ul>

</div>

<div dir="rtl" align="right">

<h3>کتابخانه‌ها</h3>

</div>

In [2]:
import numpy as np
import pandas as pd

<div dir="rtl" align="right">

<h3>۱. بارگذاری داده</h3>
<p>دو ستون <code>title</code> و <code>description</code> متن‌های سنگینی دارند. اول فقط نام ستون‌ها را می‌خوانیم، بعد این دو را از لیست حذف می‌کنیم، و CSV را با همان لیست کوتاه‌شده می‌خوانیم.</p>

</div>

In [3]:
# we load data in this cell.
# description and title columns are skipped for now

header = pd.read_csv("Divar.csv", nrows=0).columns
skip = {"description", "title"}
usecols = [c for c in header if c not in skip]
data = pd.read_csv("Divar.csv", usecols=usecols, low_memory=False)

<div dir="rtl" align="right">

<h3>۲. بررسی ابعاد</h3>
<p>تعداد (سطر، ستون). عدد ۱۰۰۰۰۰۰ سطر مورد انتظار است.</p>

</div>

In [4]:
print(data.shape)

(1000000, 59)


<div dir="rtl" align="right">

<h3>۳. نوع ستون‌ها قبل از پاک‌سازی</h3>
<p>نوع حدس‌زده‌ی pandas برای هر ستون. اعداد به‌صورت <code>float64</code>، متن‌ها به‌صورت <code>str</code> یا <code>object</code>، و هر چیز پاک‌نشده هنوز به‌صورت متن.</p>

</div>

In [5]:
data.dtypes

Unnamed: 0                      int64
cat2_slug                         str
cat3_slug                         str
city_slug                         str
neighborhood_slug                 str
created_at_month                  str
user_type                         str
rent_mode                         str
rent_value                    float64
rent_to_single                 object
rent_type                         str
price_mode                        str
price_value                   float64
credit_mode                       str
credit_value                  float64
rent_credit_transform          object
transformable_price            object
transformable_credit          float64
transformed_credit            float64
transformable_rent            float64
transformed_rent              float64
land_size                     float64
building_size                 float64
deed_type                         str
has_business_deed              object
floor                             str
rooms_count 

<div dir="rtl" align="right">

<h3>۴. حذف ستون‌های بی‌استفاده</h3>
<p>حذف <code>Unnamed: 0</code> (ایندکس اضافی) و هر ستونی که برای کمتر از ۰.۱٪ سطرها مقدار دارد.</p>

</div>

In [6]:
# we are dropping the "Unnamed: 0" column and any column that is filled for less than 0.1% of the rows.
# these columns don't give us much information and just slows down the progress
# ["rent_to_single"] only had 18 values filled

data = data.drop(columns=[c for c in ["Unnamed: 0"] if c in data], errors="ignore")

min_fill = 0.001
empty_cols = [c for c in data.columns if data[c].notna().mean() < min_fill]
data = data.drop(columns=empty_cols)
print("dropped near-empty columns:", empty_cols)

dropped near-empty columns: ['rent_to_single']


<div dir="rtl" align="right">

<h3>۵. دسته‌بندی ستون‌ها بر اساس نوع</h3>
<p>برای چند سلول بعدی به سه لیست نیاز داریم:</p>
<ul>
<li><code>numericals</code> — ستون‌هایی که عدد واقعی دارند (قیمت‌ها، متراژها، مختصات)</li>
<li><code>categoricals</code> — ستون‌های متنی (شهر، حالت قیمت، نوع سند، …)</li>
<li><code>booleans</code> — ستون‌های درست/غلط</li>
</ul>
<p>تابع <code>select_dtypes</code> این لیست‌ها را خودکار می‌سازد تا مجبور نباشیم نام ستون‌ها را دستی تایپ کنیم.</p>

</div>

In [7]:
numericals   = data.select_dtypes(include=["number"]).columns.tolist()
categoricals = data.select_dtypes(include=["str", "object", "category", "string"]).columns.tolist()
booleans     = data.select_dtypes(include=["bool", "boolean"]).columns.tolist()

<div dir="rtl" align="right">

<h3>۶. چاپ سه لیست</h3>
<p>برای دیدن این‌که چه ستون‌هایی در هر دسته افتاده‌اند. جمع سه لیست باید برابر با تعداد کل ستون‌ها باشد.</p>

</div>

In [8]:
# categorizing datas in numericals and categoricals and booleans
# we can rerun this cell a lot

print(numericals)
print(categoricals)
print(booleans)
print()
print(len(numericals))
print(len(categoricals))
print(len(booleans))
sum = len(numericals) + len(categoricals) + len(booleans)
print(sum)

['rent_value', 'price_value', 'credit_value', 'transformable_credit', 'transformed_credit', 'transformable_rent', 'transformed_rent', 'land_size', 'building_size', 'regular_person_capacity', 'cost_per_extra_person', 'rent_price_on_regular_days', 'rent_price_on_special_days', 'rent_price_at_weekends', 'location_latitude', 'location_longitude', 'location_radius']
['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'created_at_month', 'user_type', 'rent_mode', 'rent_type', 'price_mode', 'credit_mode', 'rent_credit_transform', 'transformable_price', 'deed_type', 'has_business_deed', 'floor', 'rooms_count', 'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator', 'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt', 'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool', 'has_jacuzzi', 'has_sauna', 'floor_materi

<div dir="rtl" align="right">

<h3>۷. بررسی مقادیر یکتای ستون‌های متنی</h3>
<p>قبل از نوشتن کد پاک‌سازی، مقادیر به‌هم‌ریخته را می‌شناسیم: <code>"unselect"</code>، کلمات فارسی مثل <code>"مقطوع"</code>، بازه‌های باز مثل <code>"30+"</code>، و برچسب سانسور‌شده‌ی <code>"قبل از ۱۳۷۰"</code>.</p>

</div>

In [9]:
# getting the unique values in each categorical column

for col in categoricals:
    print(col, data[col].unique())
    print(75 * "-")

cat2_slug <StringArray>
[      'temporary-rent',     'residential-sell',     'residential-rent',
      'commercial-rent',      'commercial-sell', 'real-estate-services']
Length: 6, dtype: str
---------------------------------------------------------------------------
cat3_slug <StringArray>
[                             'villa',                     'apartment-sell',
                     'apartment-rent',                        'office-rent',
                          'shop-sell',                           'plot-old',
                   'house-villa-sell',                   'house-villa-rent',
                          'shop-rent', 'industry-agriculture-business-rent',
                        'office-sell', 'industry-agriculture-business-sell',
                            'presell',                    'suite-apartment',
                        'partnership',                          'workspace',
                                  nan]
Length: 17, dtype: str
------------------------------

<div dir="rtl" align="right">

<h3>۸. بررسی ستون‌های عددی</h3>
<p>پنج مقدار اول هر ستون عددی. <code>NaN</code> یعنی آن ستون برای آن سطر خالی است — طبیعی است چون هر آگهی فقط فیلدهای مربوط به خودش را پر می‌کند.</p>

</div>

In [10]:
# getting the head for each numerical column

for col in numericals:
    print(col, data[col].head())
    print(75 * "-")

rent_value 0           NaN
1           NaN
2    26000000.0
3    95000000.0
4           NaN
Name: rent_value, dtype: float64
---------------------------------------------------------------------------
price_value 0             NaN
1    8.500000e+09
2             NaN
3             NaN
4    5.750000e+09
Name: price_value, dtype: float64
---------------------------------------------------------------------------
credit_value 0            NaN
1            NaN
2    750000000.0
3    950000000.0
4            NaN
Name: credit_value, dtype: float64
---------------------------------------------------------------------------
transformable_credit 0            NaN
1            NaN
2    750000000.0
3    950000000.0
4            NaN
Name: transformable_credit, dtype: float64
---------------------------------------------------------------------------
transformed_credit 0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: transformed_credit, dtype: float64
------------------------------------------------------

<div dir="rtl" align="right">

<h3>۹. بررسی مقادیر یکتای ستون‌های boolean</h3>
<p>مثل سلول قبلی ولی برای ستون‌های boolean، تا مطمئن شویم فقط <code>True</code>، <code>False</code> و <code>NaN</code> دارند.</p>

</div>

In [11]:
# getting the unique values in each boolean column

for col in booleans:
    print(col, data[col].unique())
    print(75 * "-")

<div dir="rtl" align="right">

<h3>۱۰. توابع کمکی</h3>
<p>دو تابع کوچک:</p>
<ul>
<li><code>fa_to_en</code> — ارقام فارسی و عربی را به ASCII تبدیل می‌کند.</li>
<li><code>to_int</code> — متن‌های به‌هم‌ریخته را به عدد صحیح: <code>"3"</code> و <code>"+30"</code> به عدد، <code>"more_than_8"</code> به ۸، <code>"قبل از ۱۳۷۰"</code> به ۱۳۶۹، و <code>"unselect"</code> به <code>NA</code>.</li>
</ul>

</div>

In [12]:
import re


FA_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩",
    "01234567890123456789",
)

def fa_to_en(x):    # converting persian/arabic numbers into english digits
    if pd.isna(x):
        return x
    return str(x).translate(FA_DIGITS)

def to_int(x):
    if pd.isna(x):
        return pd.NA

    s = fa_to_en(x).strip()
# handling missing-like values
    if s.lower() in {"unselect", "nan", "none", "", "-"}:
        return pd.NA

    pattern = r"-?\d+(?:\.\d+)?"
    #
    if "قبل از" in s:
        nums = re.findall(pattern, s)
        if nums:
            return int(float(nums[0])) - 1
    #
    if "more_than_" in s:
        nums = re.findall(pattern, s)
        if nums:
            return int(float(nums[0]))

    s = s.replace("+", "")

    match = re.search(pattern, s)
    if not match:
        return pd.NA

    return int(float(match.group()))

<div dir="rtl" align="right">

<h3>۱۱. اعمال <code>to_int</code> و ساختن ستون‌های جدید</h3>
<p>سه کار اینجا انجام می‌شود:</p>
<ol>
<li>برای هر ستون توکنی یک پرچم <code>_is_capped</code> ساخته می‌شود. <code>True</code> یعنی متن اصلی <code>"30+"</code> یا <code>"more_than_8"</code> بوده — یک بازه‌ی باز. بعد از تبدیل به عدد این اطلاعات از دست می‌رفت، پس آن را در یک ستون جدا نگه می‌داریم.</li>
<li>ستون <code>construction_year_is_censored</code> ساخته می‌شود. برچسب <code>"قبل از ۱۳۷۰"</code> یعنی «ساخته‌شده پیش از ۱۳۷۰» — هر چیزی قدیمی‌تر از آن در یک سطل جمع شده، پس هر میانگینی از سن این سطرها فقط یک کران پایین است.</li>
<li>هر ستون به <code>Int64</code> (عدد صحیح nullable) تبدیل می‌شود و <code>building_age = 1403 − construction_year</code> ساخته می‌شود. عدد <code>1403</code> سال مرجع این دیتاست است.</li>
</ol>

</div>

In [13]:
# applying (to_int) function to columns with numerical values but with string type

cols = [
    "extra_person_capacity",
    "regular_person_capacity",
    "construction_year",
    "unit_per_floor",
    "total_floors_count",
    "floor",
]

# creates new column when a value is capped. (makes flag) 
for col in cols:
    s = data[col].astype("string")
    data[f"{col}_is_capped"] = s.str.contains(r"\+|more_than", regex=True, na=False)

# makes flag for buildings before 1370
data["construction_year_is_censored"] = (
    data["construction_year"].astype("string").eq("قبل از ۱۳۷۰")
)

# applying (to_int) function to the list cols at the top.
for col in cols:
    data[col] = data[col].map(to_int).astype("Int64")

# makes a column representing building age. (Taking the data origin the year 1403)
data["building_age"] = (1403 - data["construction_year"]).astype("Int64")

<div dir="rtl" align="right">

<h3>۱۲. شاخص تورم ماهانه</h3>
<p>به هر ماه یک نرخ تورم سالانه نسبت می‌دهیم تا بعداً بتوانیم قیمت‌ها را در طول زمان مقایسه کنیم (قیمت واقعی به‌جای قیمت اسمی).</p>
<p>این لیست از amar.org.ir گرفته شده و <strong>موقتی</strong> است — باید قبل از گزارش نهایی بازبینی شود. نحوه‌ی تطبیق این‌طور است: قدیمی‌ترین ماه داده اولین نرخ، ماه بعدی دومی، و به همین ترتیب.</p>

</div>

In [14]:
inflations = [
    4.37, 3.56, 3.42, 3.29, 3.24, 3.05, 3.00, 2.95, 2.90, 2.84,
    2.79, 2.72, 2.63, 2.54, 2.45, 2.37, 2.29, 2.21, 2.14, 2.06,
    2.00, 1.94, 1.87, 1.82, 1.77, 1.71, 1.66, 1.60, 1.56, 1.51,
    1.47, 1.42, 1.38, 1.34, 1.31, 1.27, 1.23, 1.21, 1.18, 1.15,
    1.13, 1.10, 1.07, 1.05, 1.03, 1.00
]       # source: amar.org.ir 

# this function is for adding numbers from inflations list for each created_at_month value

def add_inflation_per_date(data, inflations):
    dates        = pd.to_datetime(data["created_at_month"])
    unique_dates = dates.dropna().unique()
    sorted_dates = sorted(unique_dates)

    if len(inflations) != len(sorted_dates):
        print("Error! Inflations list is not the same length as dates")

    date_to_infl = {}
    for i in range(len(sorted_dates)):
        date = sorted_dates[i]
        date_to_infl[date] = inflations[i]     # pairing each date with inflation rate

    new_column = []
    for date in dates:
        if pd.isna(date):
            new_column.append(np.nan)       # making new colum for inflation rate per date
        else:
            new_column.append(date_to_infl[date])

    data["inflation_index"] = new_column

    return data

<div dir="rtl" align="right">

<h3>۱۳. اعمال تابع تورم</h3>

</div>

In [15]:
# using the function at the top ^^^^^

data = add_inflation_per_date(data, inflations)

<div dir="rtl" align="right">

<h3>۱۴. بررسی سریع <code>inflation_index</code></h3>

</div>

In [16]:
data["inflation_index"].head()

0    1.18
1    1.27
2    1.13
3    1.23
4    1.27
Name: inflation_index, dtype: float64

<div dir="rtl" align="right">

<h3>۱۵. تبدیل متن‌های جایگزین به مقدار گم‌شده‌ی واقعی</h3>
<p>فرم دیوار وقتی کاربر چیزی انتخاب نمی‌کند <code>"unselect"</code> می‌نویسد. می‌خواهیم این‌ها <code>pd.NA</code> باشند، نه رشته‌ی <code>"unselect"</code>.</p>
<p>این حلقه روی همه‌ی ستون‌های متنی می‌رود و این توکن‌ها را با مقدار گم‌شده عوض می‌کند.</p>

</div>

In [17]:
# fixing all missing values in categorical columns

missing_likes = ["unselect", "", " ", "nan", "null", "UNSELECT", "None"]

for col in categoricals:
    data[col] = data[col].replace(missing_likes, pd.NA)

<div dir="rtl" align="right">

<h3>۱۶. ترجمه‌ی <code>user_type</code> به انگلیسی</h3>
<p>مقادیر فارسی با معادل انگلیسی جایگزین می‌شوند، و خانه‌های خالی با <code>"unknown"</code> پر می‌شوند تا بعداً بدون نگرانی از NaN بتوان از این ستون استفاده کرد.</p>

</div>

In [18]:
# changing values in user_type column to english values and filling "unknown" in empty values

data["user_type"] = data["user_type"].replace({
    "مشاور املاک": "agency",
    "شخصی" : "individual"
}).fillna("unknown")

<div dir="rtl" align="right">

<h3>۱۷. ترجمه‌ی سه ستون حالت</h3>
<p><code>rent_mode</code>، <code>price_mode</code> و <code>credit_mode</code> همه از یک واژه‌های فارسی استفاده می‌کنند:</p>
<ul>
<li><code>"مقطوع"</code> → <code>"fixed"</code></li>
<li><code>"مجانی"</code> → <code>"free"</code></li>
<li><code>"توافقی"</code> → <code>"negotiable"</code></li>
</ul>

</div>

In [19]:
# chaning the persian values in "rent_mode", "price_mode", "credit_mode" columns with english ones

mode_map = {
    "مقطوع": "fixed",
    "مجانی": "free",
    "توافقی": "negotiable",
}

for col in ["rent_mode", "price_mode", "credit_mode"]:
    data[col] = data[col].replace(mode_map)

<div dir="rtl" align="right">

<h3>۱۸. اصلاح <code>has_balcony</code></h3>
<p>این ستون به‌صورت متن ذخیره شده (<code>"true"</code>، <code>"False"</code>، <code>"unselect"</code>)، برخلاف بقیه‌ی ستون‌های <code>has_*</code> که از قبل boolean بودند. اینجا متن را به boolean واقعی تبدیل می‌کنیم تا مثل بقیه رفتار کند.</p>

</div>

In [20]:
# fixing the haS_balcony column and changing it into boolean data type

data["has_balcony"] = data["has_balcony"].replace({
    "true": True,
    "false": False,
    "True": True,
    "False": False,
    "unselect": pd.NA
}).astype("boolean")

<div dir="rtl" align="right">

<h3>۱۹. یکسان‌سازی نوع همه‌ی ستون‌های boolean</h3>
<p>بعد از مرحله‌ی ۱۸، همه‌ی ستون‌های درست/غلط قاطی شده‌اند (<code>bool</code> و <code>object</code>).</p>
<p>این حلقه همه را به نوع nullable boolean (<code>"boolean"</code>) تبدیل می‌کند که از مقدار گم‌شده‌ی واقعی پشتیبانی می‌کند.</p>

</div>

In [21]:
# changing all [True, False] columns into boolean data type

bool_cols = [
    "has_business_deed",
    "rent_credit_transform",
    "has_balcony",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "is_rebuilt",
    "has_water",
    "has_electricity",
    "has_gas",
    "has_security_guard",
    "has_barbecue",
    "has_pool",
    "has_jacuzzi",
    "has_sauna",
    "transformable_price"
]

for col in bool_cols:
    data[col] = data[col].astype("boolean")

<div dir="rtl" align="right">

<h3>۲۰. تعداد اتاق: متن → عدد</h3>
<p><code>rooms_count</code> به فارسی نوشته شده:</p>
<ul>
<li><code>"بدون اتاق"</code> → ۰</li>
<li><code>"یک"</code> … <code>"چهار"</code> → ۱ تا ۴</li>
<li><code>"پنج یا بیشتر"</code> → ۵ (یک <strong>سطل باز</strong> یعنی «۵ یا بیشتر»)</li>
</ul>
<p>این سطرها در <code>rooms_is_capped</code> علامت‌گذاری می‌شوند تا بقیه‌ی کد بداند ۵ فقط یک کران پایین است.</p>

</div>

In [22]:
# changing rooms_count string values into numerical values

rooms_map = {
    "بدون اتاق": 0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر": 5,
}

data["rooms_is_capped"] = data["rooms_count"].eq("پنج یا بیشتر")
data["rooms_count"] = data["rooms_count"].map(rooms_map).astype("Int64")

<div dir="rtl" align="right">

<h3>۲۱. دو تبدیل دیگر float→int</h3>
<p>این دو ستون قبلاً عدد بودند ولی به‌صورت <code>float64</code> ذخیره شده‌اند. تبدیل آن‌ها به <code>Int64</code> مانع می‌شود به اشتباه به‌عنوان متغیر پیوسته در نظر گرفته شوند.</p>

</div>

In [23]:
# these columns are in float64. we convert them to int64

data["regular_person_capacity"] = data["regular_person_capacity"].astype("Int64")
data["cost_per_extra_person"]   = data["cost_per_extra_person"].astype("Int64")

<div dir="rtl" align="right">

<h3>۲۲. تبدیل ماه ساخت آگهی</h3>
<p><code>created_at_month</code> یک متن مثل <code>"2024-08-01 00:00:00"</code> است. تبدیلش می‌کنیم به datetime واقعی و بعد اضافه می‌کنیم:</p>
<ul>
<li><code>created_year_g</code> — سال میلادی (برای گروه‌بندی سالانه)</li>
<li><code>created_month_g</code> — شماره‌ی ماه میلادی</li>
<li><code>created_month_fa</code> — یک برچسب خوانا شمسی مثل <code>"1403/05"</code></li>
</ul>
<p><code>jdatetime</code> اختیاری است. اگر نصب نباشد، برچسب شمسی ساخته نمی‌شود.</p>

</div>

In [24]:
# parsing the creation month
# these columns are made for Q3 and Q6 of the document

data["created_at_month"] = pd.to_datetime(data["created_at_month"], errors="coerce")
data["created_year_g"]  = data["created_at_month"].dt.year.astype("Int16")
data["created_month_g"] = data["created_at_month"].dt.month.astype("Int8")

try:
    import jdatetime
    def _jalali_label(ts):
        if pd.isna(ts):
            return pd.NA
        mid = (ts + pd.Timedelta(days=14)).date()
        j = jdatetime.date.fromgregorian(date=mid)
        return f"{j.year}/{j.month:02d}"
    data["created_month_fa"] = data["created_at_month"].map(_jalali_label).astype("string")
except ImportError:
    print("install jdatetime for Jalali labels")

<div dir="rtl" align="right">

<h3>۲۳. خانواده‌ی معامله و مقدار هدف</h3>
<p><code>cat2_slug</code> شش مقدار دارد که در چهار خانواده جمع می‌شوند:</p>
<ul>
<li><code>sell</code> (فروش مسکونی + تجاری)</li>
<li><code>long_rent</code> (رهن و اجاره‌ی ماهانه‌ی مسکونی + تجاری)</li>
<li><code>temp_rent</code> (اجاره‌ی موقت / روزانه)</li>
<li><code>service</code></li>
</ul>
<p>این گروه‌ها به رفتار قیمتی متفاوتی نیاز دارند، چون قیمت فروش و اجاره‌ی ماهانه در یک مقیاس نیستند.</p>
<p>بعد ستون <code>target_value</code> ساخته می‌شود که <strong>قیمت درست هر خانواده</strong> را نگه می‌دارد:</p>
<ul>
<li>آگهی فروش → <code>price_value</code></li>
<li>آگهی رهن و اجاره → <code>transformed_credit</code></li>
<li>آگهی اجاره‌ی موقت → <code>rent_price_on_regular_days</code></li>
</ul>
<p>از این‌جا به بعد — پاک‌سازی outlier، نمودارها، مدل‌ها — همه باید از <code>target_value</code> استفاده کنند.</p>

</div>

In [25]:
# this cell makes deal_family column.
# Used for Q4 and Q7 of document
# target_value is built later, AFTER source columns are cleaned.

deal_family_map = {
    "residential-sell": "sell",
    "commercial-sell":  "sell",
    "residential-rent": "long_rent",
    "commercial-rent":  "long_rent",
    "temporary-rent":   "temp_rent",
    "real-estate-services": "service",
}
data["deal_family"] = data["cat2_slug"].map(deal_family_map).astype("category")

<div dir="rtl" align="right">

<h3>۲۴. تبدیل ستون‌های کم‌تعداد به <code>category</code></h3>
<p>نوع <code>category</code> هر رشته‌ی یکتا را یک بار ذخیره می‌کند و برای هر سطر یک عدد کوچک نگه می‌دارد. این کار حافظه را کمتر و گروه‌بندی را سریع‌تر می‌کند.</p>
<p>فقط ستون‌هایی که حداکثر ۵۰۰۰ مقدار یکتا دارند تبدیل می‌شوند. چیزی با مقدارهای بیشتر (مثل متن آزاد یک عنوان) نه‌فقط سریع‌تر نمی‌شود، کندتر هم می‌شود.</p>

</div>

In [26]:
for col in data.select_dtypes(include=["object", "string"]).columns:
    if col in {"description", "title"}:
        continue
    if data[col].nunique(dropna=True) <= 5000:
        data[col] = data[col].astype("category")

<div dir="rtl" align="right">

<h3>۲۵. نرمال‌سازی متن فارسی</h3>
<p>متن فارسی در طبیعت شامل این‌هاست:</p>
<ul>
<li><code>ي</code> و <code>ك</code> عربی که شبیه <code>ی</code> و <code>ک</code> فارسی به نظر می‌رسند،</li>
<li>نیم‌فاصله (ZWNJ)،</li>
<li>اعراب، علامت‌های جهت، و فاصله‌های اضافی.</li>
</ul>
<p>این تابع همه را یکدست می‌کند، تا regexی که برای <code>"متر"</code> نوشته‌ایم روی متنی که <code>ي</code> عربی در همان کلمه دارد هم منطبق شود.</p>

</div>

In [27]:
# a function for making persian text manageable

def normalize_fa(s):

    s = str(s)
    s = s.replace("ي", "ی").replace("ك", "ک")
    s = s.replace("\u200c", " ")        # replaces ZWNJ with normal space
    s = re.sub(r"[\u064B-\u065F\u0670]", "", s)         # removing diacritics
    s = re.sub(r"[\u200e\u200f\u202a-\u202e]", "", s)   # removing bidi marks
    s = re.sub(r"\s+", " ", s).strip()      # makes multiple spaces into one and strips

    return s

<div dir="rtl" align="right">

<h3>۲۶. خواندن و نرمال‌سازی ستون <code>title</code></h3>
<p>هنگام بارگذاری اولیه <code>title</code> را رد کردیم تا حافظه کم مصرف شود. حالا که واقعاً به آن نیاز داریم، فقط همان یک ستون را می‌خوانیم و از <code>normalize_fa</code> عبور می‌دهیم تا regexهای بعدی قابل اعتماد باشند.</p>

</div>

In [28]:
texts = pd.read_csv("Divar.csv", usecols=["title", "description"])

data["title"]       = texts["title"].values
data["description"] = texts["description"].values

<div dir="rtl" align="right">

<h3>۲۷. ذخیره‌ی مقادیر اصلی boolean‌ها</h3>
<p>قبل از استخراج متن، مقادیر فرم دیوار را کپی می‌کنیم تا بعداً فقط جاهای خالی با متن پر شوند.</p>

</div>

In [29]:
# saving original booleans before overwriting

original_booleans = {c: data[c].copy() for c in bool_cols}

<div dir="rtl" align="right">

<h3>۲۸. استخراج boolean از متن</h3>
<p>چهار ویژگی از <code>title</code> و <code>description</code> استخراج می‌شود: <code>has_pool</code>، <code>has_sauna</code>، <code>has_jacuzzi</code>، <code>has_barbecue</code>.</p>
<p>خروجی سه حالت دارد: <code>True</code> ، <code>False</code> ، <code>NA</code>.</p>

</div>

In [30]:
import re
import pandas as pd

# ---------------------------------------------------------------------
# 1. Persian normalization
# ---------------------------------------------------------------------
_TRANS = str.maketrans({
    "ي": "ی", "ك": "ک", "ة": "ه", "ۀ": "ه",
    "أ": "ا", "إ": "ا", "ٱ": "ا", "ؤ": "و", "ئ": "ی",
    "\u064B": "", "\u064C": "", "\u064D": "",
    "\u064E": "", "\u064F": "", "\u0650": "",
    "\u0651": "", "\u0652": "",
    "\u200e": "", "\u200f": "",
    "\u202a": "", "\u202b": "", "\u202c": "", "\u202d": "", "\u202e": "",
})


def normalize_fa(s):
    if pd.isna(s):
        return ""
    s = str(s).translate(_TRANS)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


# ---------------------------------------------------------------------
# 2. Boundaries
# ---------------------------------------------------------------------
_ZWNJ = "\u200c"
_WORD = f"[\\w{_ZWNJ}]"


def _bw(word):
    """full word boundary — for short utility tokens."""
    return f"(?<!{_WORD}){word}(?!{_WORD})"


def _bp(word):
    """prefix boundary only — for nouns that take suffixes."""
    return f"(?<!{_WORD}){word}"


def _any(*words):
    return "|".join(words)


# ---------------------------------------------------------------------
# 3. Features we want only: pool, sauna, jacuzzi, barbecue
# ---------------------------------------------------------------------
FEATURE_WORDS = {
    "has_pool": [
        "استخر",
        r"حوض\s*شنا",
        "pool",
        "swimming",
    ],
    "has_sauna": [
        "سونا",
        "sauna",
    ],
    "has_jacuzzi": [
        "جکوزی",
        "جاکوزی",
        "jacuzzi",
    ],
    "has_barbecue": [
        "باربیکیو",
        "باربیکو",
        "باربکیو",
        "باربکویو",
        r"باربی\s*کیو",
        r"کباب\s*پز",
        "منقل",
        "آتشکده",
        r"آتش\s*دان",
        "آتشدان",
        "barbecue",
        "barbeque",
        "bbq",
    ],
}


def _make_positive(words):
    return _any(*[_bp(w) for w in words])


FEATURES = {col: _make_positive(words) for col, words in FEATURE_WORDS.items()}
BOOL_COLS = list(FEATURES.keys())


# ---------------------------------------------------------------------
# 4. Negation cues
# ---------------------------------------------------------------------
NEG_BEFORE = (
    r"(?:بدون|فاقد|عدم|بدون\s+وجود|ندار(?:د|ند)?|نداشته|"
    r"نمی\s*باشد|نیست|غیر\s*فعال|غیرفعال|خراب|معیوب|"
    r"از\s*کار\s*افتاده|تعطیل(?:\s*شده)?|وجود\s*ندارد|موجود\s*نیست)"
)

NEG_AFTER = (
    r"(?:ندارد|ندارند|نداشت|نداشته|نمی\s*باشد|نیست|"
    r"خراب|معیوب|از\s*کار\s*افتاده|تعطیل(?:\s*شده)?|"
    r"غیر\s*فعال|غیرفعال|وجود\s*ندارد|موجود\s*نیست)"
)

HARD_NEG_AFTER = (
    r"(?:خراب|معیوب|از\s*کار\s*افتاده|تعطیل(?:\s*شده)?|"
    r"غیر\s*فعال|غیرفعال|وجود\s*ندارد|موجود\s*نیست|"
    r"نیست|نمی\s*باشد)"
)

POS_CUE = (
    r"(?:دارد|دارند|هست|موجود|فعال|مجهز|برقرار|"
    r"قابل\s*استفاده|در\s*اختیار)"
)


def _make_other_terms(feature_key):
    other = []
    for k, words in FEATURE_WORDS.items():
        if k != feature_key:
            other.extend(words)
    return _any(*other)


def _build_patterns():
    pos = {}
    neg = {}

    for col, term in FEATURES.items():
        other = _make_other_terms(col)

        # بدون استخر / فاقد استخر / عدم وجود استخر
        neg_before = rf"(?:{NEG_BEFORE})[^.]{{0,20}}?(?:{term})"

        # استخر خراب / استخر تعطیل / استخر غیرفعال
        # other terms are blocked so "استخر دارد و جکوزی خراب" does not make pool False
        neg_after_hard = rf"(?:{term})(?:(?!{other})[^.]){{0,30}}?(?:{HARD_NEG_AFTER})"

        # استخر ندارد / استخر نیست / استخر و جکوزی ندارد
        # POS_CUE prevents "استخر دارد و جکوزی ندارد" from making pool False
        neg_after_soft = rf"(?:{term})(?:(?!{POS_CUE})[^.]){{0,25}}?(?:{NEG_AFTER})"

        neg[col] = rf"(?:{neg_before}|{neg_after_hard}|{neg_after_soft})"
        pos[col] = term

    return pos, neg


POS_PATTERNS, NEG_PATTERNS = _build_patterns()


# ---------------------------------------------------------------------
# 5. Vectorized extractor
# ---------------------------------------------------------------------
def add_bool_features(data, title_col="title", description_col="description"):
    if title_col and title_col in data.columns:
        title = data[title_col].fillna("").astype(str)
    else:
        title = pd.Series("", index=data.index, dtype="object")

    if description_col and description_col in data.columns:
        desc = data[description_col].fillna("").astype(str)
    else:
        desc = pd.Series("", index=data.index, dtype="object")

    combined = (title + " " + desc).map(normalize_fa)
    has_text = combined.str.strip().ne("")

    for col in BOOL_COLS:
        found_pos = combined.str.contains(
            POS_PATTERNS[col],
            regex=True,
            na=False,
            flags=re.IGNORECASE,
        )

        found_neg = combined.str.contains(
            NEG_PATTERNS[col],
            regex=True,
            na=False,
            flags=re.IGNORECASE,
        )

        # True only when positive keyword exists and no negation exists.
        # False when negation exists.
        # NA when feature is not mentioned.
        result = pd.Series(pd.NA, index=data.index, dtype="boolean")
        result[has_text & found_neg] = False
        result[has_text & found_pos & ~found_neg] = True

        data[col] = data[col].astype("boolean").fillna(result)

    return data

<div dir="rtl" align="right">

<h3>۲۹. اجرای استخراج</h3>
<p>تابع روی کل داده اجرا می‌شود. مقدارهای فرم دست‌نخورده می‌مانند و فقط خانه‌های خالی با متن پر می‌شوند.</p>

</div>

In [31]:
data = add_bool_features(data, title_col="title", description_col="description")

<div dir="rtl" align="right">

<h3>۳۰. بررسی سریع نتیجه</h3>
<p>نوع ستون‌ها، تعداد گم‌شده‌ها و توزیع <code>True</code> / <code>False</code> / <code>NA</code> چاپ می‌شود.</p>

</div>

In [32]:
print(data[BOOL_COLS].dtypes)
print(data[BOOL_COLS].isna().sum())
print(data[BOOL_COLS].apply(pd.Series.value_counts, dropna=False))

has_pool        boolean
has_sauna       boolean
has_jacuzzi     boolean
has_barbecue    boolean
dtype: object
has_pool        936064
has_sauna       961850
has_jacuzzi     957235
has_barbecue    960395
dtype: int64
       has_pool  has_sauna  has_jacuzzi  has_barbecue
False     26243      27277        26657         23437
True      37693      10873        16108         16168
<NA>     936064     961850       957235        960395


<div dir="rtl" align="right">

<h3>۳۱. تعریف تابع استخراج متراژ از عنوان</h3>
<p>تابع <code>extract_building_size</code> با regex عدد کنار <code>متر</code>، <code>متری</code> یا <code>متراژ</code> را پیدا می‌کند. عنوان‌هایی که به قیمت اشاره دارند رد می‌شوند. فقط مقادیر بین ۴۰ تا ۱۰۰۰ متر مربع پذیرفته می‌شوند.</p>

</div>

In [33]:
# a function for extracting land_size from title column.
# mostly using regex

def extract_building_size(text):
    if pd.isna(text):
        return pd.NA

    s = str(text).translate(FA_DIGITS)

    if any(w in s for w in ["میلیون", "میلیارد", "تومان", "تومن"]):
        return pd.NA

    if re.search(r"متری\s*از", s):
        return pd.NA

    m = re.search(r"(\d+(?:\.\d+)?)\s*(?:متراژ|متری|متر)", s)
    if m:
        v = float(m.group(1))
        return v if 40 <= v <= 1000 else pd.NA

    m = re.search(r"متراژ\s*(\d+(?:\.\d+)?)", s)
    if m:
        v = float(m.group(1))
        return v if 40 <= v <= 1000 else pd.NA

    if re.search(r"هر\s*متر", s):
        return pd.NA

    return pd.NA

<div dir="rtl" align="right">

<h3>۳۲. پر کردن <code>building_size</code> از عنوان</h3>
<p>متراژ از عنوان استخراج می‌شود و فقط جاهای خالی <code>building_size</code> پر می‌شوند. مقادیر اصلی بازنویسی نمی‌شوند. محدوده‌ی پذیرفته‌شده ۴۰ تا ۱۰۰۰ متر مربع است.</p>

</div>

In [34]:
# using the above function to extract values in extracted_land_size column
data["extracted_building_size"] = data["title"].map(extract_building_size).astype("Float64")

# filling missing values of land_size with the new column
data["building_size"] = data["building_size"].fillna(data["extracted_building_size"])


<div dir="rtl" align="right">

<h3>۳۳. حذف ستون‌های موقت</h3>
<p><code>title</code> و <code>description</code> فقط برای استخراج متن لازم بودند و حذف می‌شوند تا حافظه و سرعت بهتر شود.</p>

</div>

In [35]:
# can be used to drop the new column

data = data.drop(columns=["extracted_building_size"])

data = data.drop(columns=["title", "description"])

<div dir="rtl" align="right">

<h3>۳۴. مدیریت outlier ها</h3>
<p>در ادامه چند روش مختلف برای تشخیص و حذف مقادیر پرت تعریف و روی ستون‌های مناسب اعمال می‌شود.</p>

</div>

<div dir="rtl" align="right">

<h3>۳۵. مدیریت outlier ها</h3>
<p>در ادامه هفت روش مختلف برای تشخیص و حذف مقادیر پرت تعریف می‌شود. هر تابع یک ستون را می‌گیرد و مقدارهای خارج از مرز را به <code>NA</code> تبدیل می‌کند. سطر حذف نمی‌شود، فقط مقدارش خالی می‌شود.</p>

</div>

<div dir="rtl" align="right">

<h3>۳۶. IQR (حد توکی)</h3>
<p>مرزها را از صدک ۲۵ و ۷۵ حساب می‌کند. <code>factor</code> شدت سخت‌گیری است — کوچک‌تر یعنی سخت‌گیرتر. سطرهای <code>NA</code> دست‌نخورده می‌مانند.</p>

</div>

In [36]:
# function 1 : IQR method
def remove_outliers_iqr(df, col, factor=1.5):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    low  = Q1 - (IQR * factor)
    high = Q3 + (IQR * factor)

    mask = df[col].notna() & ~df[col].between(low, high)
    df.loc[mask, col] = pd.NA
    return df
# keeps a value if it's missing or inside the range.

<div dir="rtl" align="right">

<h3>۳۷. برش صدکی</h3>
<p>درصد مشخصی از پایین و بالای داده حذف می‌شود. پیش‌فرض ۱٪ پایین و ۱٪ بالا.</p>

</div>

In [37]:
# function 2 : Pecentile cut
def remove_outliers_percentile(df, col, low=0.01, high=0.99):
    lower  = df[col].quantile(low)
    higher = df[col].quantile(high)

    mask = df[col].notna() & ~df[col].between(lower, higher)
    df.loc[mask, col] = pd.NA
    return df
# same as IQR. keeps the missing values or values inside range.

<div dir="rtl" align="right">

<h3>۳۸. Z-score</h3>
<p>فاصله‌ی هر مقدار از میانگین را به واحد انحراف معیار می‌سنجد. <code>threshold</code> پیش‌فرض ۳ است. برای قیمت مناسب نیست چون میانگین و انحراف معیار خودشان با outlier خراب می‌شوند.</p>

</div>

In [38]:
# function 3 : Z-score
def remove_outliers_zscore(df, col, threshold=3):
    mean = df[col].mean()
    std  = df[col].std()
    z    = ((df[col] - mean) / std)

    mask = df[col].notna() & (z.abs() >= threshold)
    df.loc[mask, col] = pd.NA
    return df
# keeps the missing values or values meeting criteria.

<div dir="rtl" align="right">

<h3>۳۹. Z-score اصلاح‌شده</h3>
<p>مثل Z-score ولی با میانه و MAD، که نسبت به outlierها مقاوم‌اند. ضریب <code>0.6745</code> نتیجه را با Z-score استاندارد قابل‌مقایسه می‌کند.</p>

</div>

In [39]:
# function 4 : modified Z-score (median_based)
def remove_outliers_mad(df, col, threshold=3.5):
    median = df[col].median()
    mad    = (df[col] - median).abs().median()

    if mad == 0:
        return df

    modified_z = 0.6745 * (df[col] - median) / mad     # using 0.6745 makes result comparable to zscore func
    mask = df[col].notna() & (modified_z.abs() >= threshold)
    df.loc[mask, col] = pd.NA
    return df
# keeps the missing values or values meeting criteria.

<div dir="rtl" align="right">

<h3>۴۰. IQR در فضای لگاریتمی</h3>
<p>قیمت‌ها چند مرتبه‌ی بزرگی متفاوتند، پس اول لگاریتم می‌گیریم تا توزیع نرمال شود و بعد IQR اعمال می‌کنیم. مقادیر صفر و منفی به <code>NA</code> تبدیل می‌شوند.</p>

</div>

In [40]:
# function 5 : log-space IQR
def remove_outliers_log_iqr(df, col, factor=1.5):
    positive = df[col].where(df[col] > 0)
    log_col  = np.log10(positive)

    Q1 = log_col.quantile(0.25)
    Q3 = log_col.quantile(0.75)
    IQR = Q3 - Q1
    low  = Q1 - (IQR * factor)
    high = Q3 + (IQR * factor) 

    out_of_bounds = log_col.notna() & ~log_col.between(low, high)
    non_positive  = df[col].notna() & (df[col] <= 0)

    mask = out_of_bounds | non_positive
    df.loc[mask, col] = pd.NA
    return df
# keeps the missing values or values meeting criteria.

<div dir="rtl" align="right">

<h3>۴۱. IQR گروهی</h3>
<p>داخل هر گروه جداگانه IQR اعمال می‌شود، چون قیمت تهران و یک روستا هم‌مقیاس نیستند. <code>group_col</code> مثل <code>city_slug</code> یا <code>deal_family</code> است.</p>

</div>

In [41]:
def remove_outliers_group_iqr(df, col, group_col, factor=1.5):
    g = df.groupby(group_col, observed=True)[col]

    Q1  = g.transform(lambda s: s.quantile(0.25))
    Q3  = g.transform(lambda s: s.quantile(0.75))
    IQR = Q3 - Q1

    low  = Q1 - factor * IQR
    high = Q3 + factor * IQR

    mask = df[col].notna() & ~df[col].between(low, high)
    df.loc[mask, col] = pd.NA
    return df

<div dir="rtl" align="right">

<h3>۴۲. پاک‌سازی متراژ</h3>
<p>دو مرحله: اول مرز سخت (۲۰ تا ۲۵۰۰ متر مربع) برای حذف مقادیر غیرممکن، بعد مرز نرم بر اساس IQR در فضای لگاریتمی با ضریب ۲.</p>

</div>

In [42]:
# function 7 : only for building size and land size
def clean_building_size(df, col='building_size', min_limit=20, max_limit=2500):
    hard_mask = df[col].notna() & ~df[col].between(min_limit, max_limit)
    df.loc[hard_mask, col] = pd.NA

    positive = df[col].where(df[col] > 0)
    log_vals = np.log10(positive)

    q1 = log_vals.quantile(0.25)
    q3 = log_vals.quantile(0.75)
    iqr = q3 - q1
    
    low = 10 ** (q1 - 2.0 * iqr)
    high = 10 ** (q3 + 2.0 * iqr)
    
    soft_mask = df[col].notna() & ~df[col].between(low, high)
    df.loc[soft_mask, col] = pd.NA
    return df

<div dir="rtl" align="right">

<h3>۴۳. نمونه‌ی فراخوانی توابع</h3>
<p>این سلول فقط چند مثال کامنت‌شده است تا الگوی استفاده نشان داده شود.</p>

</div>

In [43]:
# this part is an example of how you call these outlier functions!

# data = remove_outliers_iqr(data, "target_value")
# data = remove_outliers_percentile(data, "building_size", low=0.05, high=0.95)
# data = remove_outliers_group_iqr(data, "target_value", "deal_family")

<div dir="rtl" align="right">

<h3>۴۴. اعمال روی ستون‌های قیمت</h3>
<p>روی شش ستون قیمت، IQR لگاریتمی با <code>factor=2.0</code> اعمال می‌شود. ضریب ۲ کمی ملایم‌تر از پیش‌فرض است تا مقادیر بزرگ ولی معتبر حذف نشوند.</p>

</div>

In [44]:
data = remove_outliers_log_iqr(data, "price_value", factor=2.0)
data = remove_outliers_log_iqr(data, "rent_value", factor=2.0)
data = remove_outliers_log_iqr(data, "credit_value", factor=2.0)
data = remove_outliers_log_iqr(data, "transformed_credit", factor=2.0)
data = remove_outliers_log_iqr(data, "transformed_rent", factor=2.0)
data = remove_outliers_log_iqr(data, "rent_price_on_regular_days", factor=2.0)

<div dir="rtl" align="right">

<h3>۴۵. اعمال روی متراژ</h3>
<p>ستون‌های <code>building_size</code> و <code>land_size</code> با تابع <code>clean_building_size</code> پاک‌سازی می‌شوند.</p>

</div>

In [45]:
data = clean_building_size(data, col="building_size")
data = clean_building_size(data, "land_size")

<div dir="rtl" align="right">

<h3>۴۶. اعمال روی تعداد طبقه و ظرفیت</h3>
<p>پنج ستون شمارشی با IQR معمولی و <code>factor=3.0</code> پاک‌سازی می‌شوند. ضریب ۳ یعنی فقط مقادیر خیلی دور حذف شوند.</p>

</div>

In [46]:
data = remove_outliers_iqr(data, "floor", factor=3.0)
data = remove_outliers_iqr(data, "total_floors_count", factor=3.0)
data = remove_outliers_iqr(data, "regular_person_capacity", factor=3.0)
data = remove_outliers_iqr(data, "extra_person_capacity", factor=3.0)
data = remove_outliers_iqr(data, "building_age", factor=3.0)

<div dir="rtl" align="right">

<h3>۴۷. اعمال روی قیمت آخر هفته</h3>
<p>دو ستون اجاره‌ی روزهای خاص و آخر هفته با IQR لگاریتمی پاک‌سازی می‌شوند.</p>

</div>

In [47]:
data = remove_outliers_log_iqr(data, "rent_price_at_weekends", factor=2.0)
data = remove_outliers_log_iqr(data, "rent_price_on_special_days", factor=2.0)

<div dir="rtl" align="right">

<h3>۴۸. ساخت <code>target_value</code></h3>
<p>حالا که ستون‌های قیمت پاک‌سازی شده‌اند، یک ستون واحد می‌سازیم: فروش از <code>price_value</code>، رهن و اجاره از <code>transformed_credit</code>، اجاره‌ی موقت از <code>rent_price_on_regular_days</code>.</p>

</div>

In [48]:
# build target_value from the already-cleaned source columns

data["target_value"] = np.nan

m = data["deal_family"] == "sell"
data.loc[m, "target_value"] = data.loc[m, "price_value"]

m = data["deal_family"] == "long_rent"
data.loc[m, "target_value"] = data.loc[m, "transformed_credit"]

m = data["deal_family"] == "temp_rent"
data.loc[m, "target_value"] = data.loc[m, "rent_price_on_regular_days"]

<div dir="rtl" align="right">

<h3>۴۹. پاک‌سازی outlier روی <code>target_value</code></h3>
<p>چون قیمت فروش و اجاره در یک مقیاس نیستند، IQR جداگانه داخل هر <code>deal_family</code> اجرا می‌شود.</p>

</div>

In [49]:
data = remove_outliers_group_iqr(data, "target_value", "deal_family", factor=2.0)

<div dir="rtl" align="right">

<h3>۵۰. فیلتر مختصات جغرافیایی</h3>
<p>مختصات خارج از مرزهای ایران حذف می‌شوند: عرض بین ۲۴ تا ۴۰.۵ و طول بین ۴۴ تا ۶۳.۵. مقادیر <code>NA</code> حفظ می‌شوند.</p>

</div>

In [50]:
data = data[data["location_latitude"].isna() | data["location_latitude"].between(24.0, 40.5)]
data = data[data["location_longitude"].isna() | data["location_longitude"].between(44.0, 63.5)]

<div dir="rtl" align="right">

<h3>۵۱. ذخیره‌ی فایل خروجی</h3>
<p>داده‌ی تمیزشده در <code>cleaned_data.csv</code> ذخیره می‌شود. <code>index=False</code> مانع نوشتن ایندکس به‌عنوان ستون اضافه می‌شود.</p>

</div>

In [51]:
# for making csv file

data.to_csv("cleaned_data.csv", index=False)

<div dir="rtl" align="right">

<h3>۵۲. جدول خلاصه‌ی عددی</h3>
<p>برای هر ستون عددی این‌ها چاپ می‌شود:</p>
<ul>
<li>تعداد مقادیر گم‌شده</li>
<li>میانگین، انحراف معیار، میانه، کمینه، بیشینه</li>
<li>Q25، Q75، IQR</li>
<li>تعداد مقادیر بیرون از حد توکی (۱.۵ × IQR)</li>
<li>چولگی و کشیدگی</li>
</ul>
<p>این جدول برای مقایسه‌ی توزیع ستون‌ها بعد از پاک‌سازی استفاده می‌شود.</p>

</div>

In [52]:
# a function for getting information about numerical columns (PASTED)
def numerical_summary(data):
    # Select only numeric columns
    num_df = data.select_dtypes(include=[np.number])
    
    if num_df.empty:
        return "No numerical columns found in the DataFrame."
    
    summary_data = []
    
    for col in num_df.columns:
        series = num_df[col].dropna()
        
        # Basic counts
        total_count = len(data[col])
        null_count = data[col].isnull().sum()
        null_pct = (null_count / total_count) * 100
        zero_count = (data[col] == 0).sum()
        zero_pct = (zero_count / total_count) * 100
        unique_count = data[col].nunique()
        
        if len(series) > 0:
            # Measures of Central Tendency & Dispersion
            mean_val = series.mean()
            std_val = series.std()
            median_val = series.median()
            min_val = series.min()
            max_val = series.max()
            range_val = max_val - min_val
            
            # Percentiles & IQR
            q25 = series.quantile(0.25)
            q75 = series.quantile(0.75)
            iqr = q75 - q25
            
            # Outlier detection (Tukey's Fences: 1.5 * IQR)
            lower_bound = q25 - (1.5 * iqr)
            upper_bound = q75 + (1.5 * iqr)
            outliers_count = ((series < lower_bound) | (series > upper_bound)).sum()
            outliers_pct = (outliers_count / len(series)) * 100
            
            # Shape of distribution
            skewness = series.skew()
            kurt = series.kurtosis()
        else:
            mean_val = std_val = median_val = min_val = max_val = range_val = np.nan
            q25 = q75 = iqr = outliers_count = outliers_pct = skewness = kurt = np.nan
            
        summary_data.append({
            'Column': col,
            'Data Type': data[col].dtype,
            'Non-Null Count': len(series),
            'Missing Count': null_count,
            'Missing (%)': round(null_pct, 2),
            'Zero Count': zero_count,
            'Zero (%)': round(zero_pct, 2),
            'Unique Values': unique_count,
            'Mean': round(mean_val, 3),
            'Std Dev': round(std_val, 3),
            'Median (Q2)': round(median_val, 3),
            'Min': round(min_val, 3),
            'Q25': round(q25, 3),
            'Q75': round(q75, 3),
            'Max': round(max_val, 3),
            'Range': round(range_val, 3),
            'IQR': round(iqr, 3),
            'Outliers (Count)': outliers_count,
            'Outliers (%)': round(outliers_pct, 2),
            'Skewness': round(skewness, 3),
            'Kurtosis': round(kurt, 3)
        })
        
    return pd.DataFrame(summary_data).set_index('Column')

# Usage:
summary_df = numerical_summary(data)
print(summary_df.T)  # Transposed for easier reading if you have many metrics


Column              rent_value     price_value   credit_value  \
Data Type              float64         float64        float64   
Non-Null Count          261183          547121         341368   
Missing Count           738793          452855         658608   
Missing (%)              73.88           45.29          65.86   
Zero Count                   0               0              0   
Zero (%)                   0.0             0.0            0.0   
Unique Values             1016           10768           1102   
Mean               17279971.49  5605797794.796   489973189.75   
Std Dev           32437294.567  8991768103.674  865580831.171   
Median (Q2)          7500000.0    2900000000.0    250000000.0   
Min                   120000.0      80000000.0      4000000.0   
Q25                  4000000.0    1500000000.0    100000000.0   
Q75                 16000000.0    5940000000.0    500000000.0   
Max                375000000.0  104500000000.0  12500000000.0   
Range              374880